# Alert Production Metrics

Explores DM pipeline alert statistics from two sources:
- **Sasquatch (InfluxDB)**: real-time prompt processing metrics per detector/visit
- **DP2 DRP (Butler)**: full reprocessing results via parquet cache

**Sections:**
1. Single night — per-detector and focal plane views (Sasquatch)
2. DIA sources — all nights, Prompt vs DP2 comparison
3. Solar system objects — all nights, Prompt vs DP2 comparison

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.collections import PatchCollection
from matplotlib.colors import Normalize
from astropy.time import Time

from rubin_nights.influx_query import InfluxQueryClient

os.environ["PGPASSFILE"] = os.path.expanduser("~/.lsst/postgres-credentials.txt")
os.environ["PGUSER"] = "rubin"

## Section 1: Single Night (Sasquatch Prompt Processing)

In [ ]:
client = InfluxQueryClient("usdf-dev", db_name="lsst.prompt")

day_obs = 20260526
date_str = f"{str(day_obs)[:4]}-{str(day_obs)[4:6]}-{str(day_obs)[6:8]}"
t_start = Time(f"{date_str}T00:00:00", scale="utc")
t_end = Time(f"{date_str}T14:00:00", scale="utc")
print(f"Night {date_str}: {t_start.isot} → {t_end.isot}")

In [ ]:
dia_det = client.select_time_series(
    "lsst.prompt.prod.numDiaSourcesGood",
    ["visit", "detector", "numAllDiaSources", "numGoodDiaSources", "run"],
    t_start,
    t_end,
)
if len(dia_det) > 0:
    dia_det["visit"] = dia_det["visit"].astype(int)
    dia_det["detector"] = dia_det["detector"].astype(int)

print(f"{len(dia_det)} detector records from {dia_det['visit'].nunique()} visits")
dia_det.head()

In [ ]:
if len(dia_det) > 0:
    visit_summary = dia_det.groupby("visit").agg(
        numAllDiaSources_sum=("numAllDiaSources", "sum"),
        numGoodDiaSources_sum=("numGoodDiaSources", "sum"),
        numGoodDiaSources_median=("numGoodDiaSources", "median"),
        nDetectors=("detector", "count"),
    )
    visit_summary.index = visit_summary.index.astype(int)
    visit_summary = visit_summary.sort_index()
    print(f"{len(visit_summary)} visits")
    display(visit_summary.describe())
    display(visit_summary.head(20))
else:
    print("No DIA data.")

In [ ]:
sso_det = client.select_time_series(
    "lsst.prompt.prod.numSsObjects",
    ["visit", "detector", "NumSsObjectsMetric", "run"],
    t_start,
    t_end,
)
dss_det = client.select_time_series(
    "lsst.prompt.prod.numDirectSsObjects",
    ["visit", "detector", "NumSsObjectsMetric", "run"],
    t_start,
    t_end,
)
print(f"SS Objects: {len(sso_det)} detector records")
print(f"Direct SS associations: {len(dss_det)} detector records")

In [ ]:
if len(dia_det) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))

    ax = axes[0, 0]
    ax.plot(
        visit_summary.index, visit_summary["numAllDiaSources_sum"], ".", ms=4, alpha=0.7
    )
    ax.set_xlabel("Visit")
    ax.set_ylabel("Total DIA Sources")
    ax.set_title(f"All DIA Sources per Visit — night {day_obs}")

    ax = axes[0, 1]
    ax.plot(
        visit_summary.index,
        visit_summary["numGoodDiaSources_sum"],
        ".",
        ms=4,
        alpha=0.7,
        color="C1",
    )
    ax.set_xlabel("Visit")
    ax.set_ylabel("Good DIA Sources")
    ax.set_title("Good DIA Sources per Visit")

    ax = axes[1, 0]
    ax.plot(
        visit_summary.index,
        visit_summary["nDetectors"],
        ".",
        ms=4,
        alpha=0.7,
        color="C2",
    )
    ax.set_xlabel("Visit")
    ax.set_ylabel("N Detectors")
    ax.set_title("Detectors processed per Visit")
    ax.axhline(189, color="gray", ls="--", alpha=0.5, label="189 (full focal plane)")
    ax.legend()

    ax = axes[1, 1]
    ax.hist(dia_det["numGoodDiaSources"].dropna(), bins=50, alpha=0.7, color="C3")
    ax.set_xlabel("Good DIA Sources per Detector")
    ax.set_ylabel("Count")
    ax.set_title("Distribution across detectors")

    plt.tight_layout()
    plt.show()
else:
    print("No data.")

In [ ]:
# Build LSSTCam focal plane geometry (21 science rafts, 9 sensors each)
rafts = [
    "R01",
    "R02",
    "R03",
    "R10",
    "R11",
    "R12",
    "R13",
    "R14",
    "R20",
    "R21",
    "R22",
    "R23",
    "R24",
    "R30",
    "R31",
    "R32",
    "R33",
    "R34",
    "R41",
    "R42",
    "R43",
]
raft_positions = {
    "R01": (1, 0),
    "R02": (2, 0),
    "R03": (3, 0),
    "R10": (0, 1),
    "R11": (1, 1),
    "R12": (2, 1),
    "R13": (3, 1),
    "R14": (4, 1),
    "R20": (0, 2),
    "R21": (1, 2),
    "R22": (2, 2),
    "R23": (3, 2),
    "R24": (4, 2),
    "R30": (0, 3),
    "R31": (1, 3),
    "R32": (2, 3),
    "R33": (3, 3),
    "R34": (4, 3),
    "R41": (1, 4),
    "R42": (2, 4),
    "R43": (3, 4),
}
sensor_offsets = [
    (-1, -1),
    (0, -1),
    (1, -1),
    (-1, 0),
    (0, 0),
    (1, 0),
    (-1, 1),
    (0, 1),
    (1, 1),
]
det_xy = {}
for ri, rn in enumerate(rafts):
    rx, ry = raft_positions[rn]
    for si, (sx, sy) in enumerate(sensor_offsets):
        det_xy[ri * 9 + si] = (rx * 3 + sx, ry * 3 + sy)

if len(dia_det) > 0:
    example_visit = visit_summary["nDetectors"].idxmax()
    det_data = dia_det[dia_det["visit"] == example_visit].set_index("detector")

    norm = Normalize(vmin=0, vmax=det_data["numGoodDiaSources"].quantile(0.95))
    cmap = plt.cm.viridis
    patches, colors = [], []
    for det_id, (x, y) in det_xy.items():
        patches.append(Rectangle((x - 0.45, y - 0.45), 0.9, 0.9))
        if det_id in det_data.index:
            colors.append(cmap(norm(det_data.loc[det_id, "numGoodDiaSources"])))
        else:
            colors.append((0.85, 0.85, 0.85, 1.0))

    fig, ax = plt.subplots(figsize=(10, 10))
    pc = PatchCollection(patches, facecolors=colors, edgecolors="k", linewidths=0.3)
    ax.add_collection(pc)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.02).set_label("Good DIA Sources")
    ax.set_xlim(-2, 15)
    ax.set_ylim(-2, 15)
    ax.set_aspect("equal")
    ax.set_xlabel("Focal Plane X")
    ax.set_ylabel("Focal Plane Y")
    ax.set_title(
        f"Visit {example_visit}: Good DIA Sources — Focal Plane Map\n"
        f"({len(det_data)} detectors with data, gray = no data)"
    )
    plt.tight_layout()
    plt.show()
else:
    print("No data.")

## Section 2: DIA Sources — All Nights

Compares prompt processing (Sasquatch) and full DRP reprocessing (DP2 Butler).  
DP2 cache built from `dia_source_visit` parquet files; reads only the `reliability` column.

In [ ]:
all_dia = client.select_time_series(
    "lsst.prompt.prod.numDiaSourcesGood",
    ["visit", "detector", "numGoodDiaSources", "run"],
    Time("2026-02-01T00:00:00", scale="utc"),
    Time("2026-05-28T00:00:00", scale="utc"),
)
all_dia["visit"] = all_dia["visit"].astype(int)
all_dia["detector"] = all_dia["detector"].astype(int)
all_dia = all_dia[all_dia["run"].str.startswith("LSSTCam/")].drop(columns=["run"])

all_visit = all_dia.groupby("visit").agg(
    numGoodDiaSources_sum=("numGoodDiaSources", "sum"),
    nDetectors=("detector", "count"),
    timestamp=("numGoodDiaSources", lambda x: x.index[0]),
)
all_visit["timestamp"] = pd.to_datetime(all_visit["timestamp"])
all_visit["day_obs"] = (all_visit.index // 100000).astype(int)
all_visit = all_visit.sort_index()

nightly = all_visit.groupby("day_obs").agg(
    numGoodDiaSources_total=("numGoodDiaSources_sum", "sum"),
    numGoodDiaSources_median=("numGoodDiaSources_sum", "median"),
    nVisits=("numGoodDiaSources_sum", "count"),
)
nightly["date"] = pd.to_datetime(nightly.index.astype(str), format="%Y%m%d")

print(
    f"{len(all_visit)} visits across {all_visit['day_obs'].nunique()} nights "
    f"({all_dia.index.min().date()} → {all_dia.index.max().date()})"
)

In [ ]:
CACHE_DIR = Path("../data/dp2_cache")
CACHE_PATH = Path("../data/dp2_dia_source_visit_counts.parquet")
N_SHARDS = 9
RELIABILITY_CUT = 0.5
FORCE_REFETCH = False  # Set True to rebuild

if FORCE_REFETCH:
    for f in CACHE_DIR.glob("shard_*.parquet"):
        f.unlink()
    if CACHE_PATH.exists():
        CACHE_PATH.unlink()
    print("Cleared DIA cache.")

if CACHE_PATH.exists():
    dp2_visits = pd.read_parquet(CACHE_PATH)
    print(f"Loaded from cache: {len(dp2_visits)} visits")
else:
    from lsst.daf.butler import Butler

    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    b = Butler("/sdf/group/rubin/repo/dp2_prep", collections="LSSTCam/runs/DRP/DP2")
    refs = list(b.registry.queryDatasets("dia_source_visit"))
    print(f"{len(refs)} files → {N_SHARDS} shards")

    shard_size = len(refs) // N_SHARDS + 1
    shards_needed = [
        s
        for s in range(N_SHARDS)
        if not (CACHE_DIR / f"shard_{s:02d}.parquet").exists()
    ]

    if shards_needed:
        sid = shards_needed[0]
        shard_refs = refs[sid * shard_size : min((sid + 1) * shard_size, len(refs))]
        print(f"Building shard {sid}/{N_SHARDS-1} ({len(shard_refs)} files)...")
        results = []
        for i, ref in enumerate(shard_refs):
            try:
                tbl = pq.read_table(b.getURI(ref).ospath, columns=["reliability"])
                rel = tbl.column("reliability").to_numpy()
                results.append(
                    {
                        "visit": ref.dataId["visit"],
                        "day_obs": ref.dataId["day_obs"],
                        "band": ref.dataId["band"],
                        "n_dia_sources": len(rel),
                        "n_high_reliability": int((rel >= RELIABILITY_CUT).sum()),
                    }
                )
            except Exception:
                pass
            if (i + 1) % 1000 == 0:
                print(f"  ...{i+1}/{len(shard_refs)}")
        pd.DataFrame(results).to_parquet(CACHE_DIR / f"shard_{sid:02d}.parquet")
        print(f"  Saved shard {sid} ({len(results)} visits)")
        shards_needed = shards_needed[1:]

    if shards_needed:
        print(f"\n⚠️  {len(shards_needed)} shards remaining — re-run to continue.")
    else:
        dp2_visits = pd.concat(
            [
                pd.read_parquet(CACHE_DIR / f"shard_{s:02d}.parquet")
                for s in range(N_SHARDS)
            ],
            ignore_index=True,
        )
        dp2_visits.to_parquet(CACHE_PATH)
        print(f"Cache complete: {len(dp2_visits)} visits")

if CACHE_PATH.exists():
    dp2_visits["date"] = pd.to_datetime(
        dp2_visits["day_obs"].astype(str), format="%Y%m%d"
    )
    dp2_nightly = dp2_visits.groupby("day_obs").agg(
        n_dia_sources_total=("n_dia_sources", "sum"),
        n_high_reliability_total=("n_high_reliability", "sum"),
        n_dia_sources_median=("n_dia_sources", "median"),
        n_high_reliability_median=("n_high_reliability", "median"),
        nVisits=("visit", "count"),
    )
    dp2_nightly["date"] = pd.to_datetime(dp2_nightly.index.astype(str), format="%Y%m%d")
    dp2_nightly["high_rel_fraction"] = (
        dp2_nightly["n_high_reliability_total"] / dp2_nightly["n_dia_sources_total"]
    )
    print(
        f"{dp2_visits['visit'].nunique()} visits, {dp2_visits['day_obs'].nunique()} nights"
    )
    print(
        f"Total: {dp2_visits['n_dia_sources'].sum():,}  "
        f"High-rel: {dp2_visits['n_high_reliability'].sum():,} "
        f"({100*dp2_nightly['high_rel_fraction'].mean():.1f}% avg)"
    )

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.scatter(
    all_visit["timestamp"],
    all_visit["numGoodDiaSources_sum"],
    s=5,
    alpha=0.4,
    label="Prompt: good DIA",
    color="C0",
)
ax.scatter(
    dp2_visits["date"],
    dp2_visits["n_dia_sources"],
    s=5,
    alpha=0.3,
    label="DP2: all DIA",
    color="C1",
)
ax.scatter(
    dp2_visits["date"],
    dp2_visits["n_high_reliability"],
    s=5,
    alpha=0.4,
    label=f"DP2: high-rel (≥{RELIABILITY_CUT})",
    color="C2",
)
ax.set_xlabel("Date")
ax.set_ylabel("DIA Sources per Visit")
ax.set_title("DIA Sources per Visit — Prompt Processing vs DP2 DRP")
ax.set_yscale("symlog", linthresh=100)
ax.legend(markerscale=3)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True)

ax = axes[0]
ax.scatter(
    nightly["date"],
    nightly["numGoodDiaSources_total"],
    s=15,
    alpha=0.7,
    label="Prompt: good DIA",
    color="C0",
)
ax.scatter(
    dp2_nightly["date"],
    dp2_nightly["n_dia_sources_total"],
    s=15,
    alpha=0.7,
    label="DP2: all DIA",
    color="C1",
)
ax.scatter(
    dp2_nightly["date"],
    dp2_nightly["n_high_reliability_total"],
    s=15,
    alpha=0.7,
    label=f"DP2: high-rel (≥{RELIABILITY_CUT})",
    color="C2",
)
ax.set_ylabel("Total DIA Sources")
ax.set_title("DIA Sources by Night — Prompt Processing vs DP2 DRP")
ax.legend()

ax = axes[1]
ax.scatter(
    nightly["date"],
    nightly["numGoodDiaSources_median"],
    s=15,
    alpha=0.7,
    label="Prompt: good DIA",
    color="C0",
)
ax.scatter(
    dp2_nightly["date"],
    dp2_nightly["n_dia_sources_median"],
    s=15,
    alpha=0.7,
    label="DP2: all DIA",
    color="C1",
)
ax.scatter(
    dp2_nightly["date"],
    dp2_nightly["n_high_reliability_median"],
    s=15,
    alpha=0.7,
    label=f"DP2: high-rel (≥{RELIABILITY_CUT})",
    color="C2",
)
ax.set_ylabel("Median DIA Sources / Visit")
ax.legend()

ax = axes[2]
ax.scatter(
    dp2_nightly["date"], dp2_nightly["high_rel_fraction"], s=15, alpha=0.7, color="C3"
)
ax.set_ylabel("High-Reliability Fraction (DP2)")
ax.set_xlabel("Night (day_obs)")
ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

## Section 3: Solar System Objects — All Nights

Compares prompt processing SS matches (Sasquatch) against DP2 ephemeris predictions.  
DP2 cache built from `preloaded_ss_object_visit`; row count = predicted SS objects in field.

In [ ]:
SS_CACHE_DIR = Path("../data/dp2_ss_cache")
SS_CACHE_PATH = Path("../data/dp2_ss_object_visit_counts.parquet")
SS_N_SHARDS = 6
SS_FORCE_REFETCH = False  # Set True to rebuild

if SS_FORCE_REFETCH:
    for f in SS_CACHE_DIR.glob("shard_*.parquet"):
        f.unlink()
    if SS_CACHE_PATH.exists():
        SS_CACHE_PATH.unlink()
    print("Cleared SS cache.")

if SS_CACHE_PATH.exists():
    dp2_ss = pd.read_parquet(SS_CACHE_PATH)
    print(f"Loaded from cache: {len(dp2_ss)} visits")
else:
    from lsst.daf.butler import Butler

    SS_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    b_ss = Butler("/sdf/group/rubin/repo/dp2_prep", collections="LSSTCam/runs/DRP/DP2")
    ss_refs = list(b_ss.registry.queryDatasets("preloaded_ss_object_visit"))
    print(f"{len(ss_refs)} files → {SS_N_SHARDS} shards")

    shard_size = len(ss_refs) // SS_N_SHARDS + 1
    shards_needed = [
        s
        for s in range(SS_N_SHARDS)
        if not (SS_CACHE_DIR / f"shard_{s:02d}.parquet").exists()
    ]

    if shards_needed:
        sid = shards_needed[0]
        shard_refs = ss_refs[
            sid * shard_size : min((sid + 1) * shard_size, len(ss_refs))
        ]
        print(f"Building shard {sid}/{SS_N_SHARDS-1} ({len(shard_refs)} files)...")
        results = []
        for i, ref in enumerate(shard_refs):
            try:
                meta = pq.read_metadata(b_ss.getURI(ref).ospath)
                results.append(
                    {
                        "visit": ref.dataId["visit"],
                        "day_obs": ref.dataId["day_obs"],
                        "band": ref.dataId["band"],
                        "n_ss_objects": meta.num_rows,
                    }
                )
            except Exception:
                pass
            if (i + 1) % 500 == 0:
                print(f"  ...{i+1}/{len(shard_refs)}")
        pd.DataFrame(results).to_parquet(SS_CACHE_DIR / f"shard_{sid:02d}.parquet")
        print(f"  Saved shard {sid} ({len(results)} visits)")
        shards_needed = shards_needed[1:]

    if shards_needed:
        print(f"\n⚠️  {len(shards_needed)} shards remaining — re-run to continue.")
    else:
        dp2_ss = pd.concat(
            [
                pd.read_parquet(SS_CACHE_DIR / f"shard_{s:02d}.parquet")
                for s in range(SS_N_SHARDS)
            ],
            ignore_index=True,
        )
        dp2_ss.to_parquet(SS_CACHE_PATH)
        print(f"Cache complete: {len(dp2_ss)} visits")

if SS_CACHE_PATH.exists():
    dp2_ss["date"] = pd.to_datetime(dp2_ss["day_obs"].astype(str), format="%Y%m%d")
    dp2_ss_nightly = dp2_ss.groupby("day_obs").agg(
        n_ss_total=("n_ss_objects", "sum"),
        n_ss_median=("n_ss_objects", "median"),
        nVisits=("visit", "count"),
    )
    dp2_ss_nightly["date"] = pd.to_datetime(
        dp2_ss_nightly.index.astype(str), format="%Y%m%d"
    )
    print(
        f"{dp2_ss['visit'].nunique()} visits, {dp2_ss['day_obs'].nunique()} nights, "
        f"median {dp2_ss['n_ss_objects'].median():.0f} SS objects/visit"
    )

In [ ]:
all_ss = client.select_time_series(
    "lsst.prompt.prod.numSsObjects",
    ["visit", "detector", "NumSsObjectsMetric", "run"],
    Time("2026-02-01T00:00:00", scale="utc"),
    Time("2026-05-28T00:00:00", scale="utc"),
)
all_ss["visit"] = all_ss["visit"].astype(int)
all_ss["detector"] = all_ss["detector"].astype(int)
all_ss = all_ss[all_ss["run"].str.startswith("LSSTCam/")].drop(columns=["run"])

all_ss_visit = all_ss.groupby("visit").agg(
    n_ss_sum=("NumSsObjectsMetric", "sum"),
    n_ss_median=("NumSsObjectsMetric", "median"),
    nDetectors=("detector", "count"),
    timestamp=("NumSsObjectsMetric", lambda x: x.index[0]),
)
all_ss_visit["timestamp"] = pd.to_datetime(all_ss_visit["timestamp"])
all_ss_visit["day_obs"] = (all_ss_visit.index // 100000).astype(int)
all_ss_visit = all_ss_visit.sort_index()

ss_nightly = all_ss_visit.groupby("day_obs").agg(
    n_ss_total=("n_ss_sum", "sum"),
    n_ss_median=("n_ss_sum", "median"),
    nVisits=("n_ss_sum", "count"),
)
ss_nightly["date"] = pd.to_datetime(ss_nightly.index.astype(str), format="%Y%m%d")

print(
    f"{len(all_ss_visit)} visits, {all_ss_visit['day_obs'].nunique()} nights "
    f"({all_ss.index.min().date()} → {all_ss.index.max().date()})"
)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.scatter(
    dp2_ss["date"],
    dp2_ss["n_ss_objects"],
    s=5,
    alpha=0.3,
    label="DP2: predicted in field",
    color="C1",
)
ax.scatter(
    all_ss_visit["timestamp"],
    all_ss_visit["n_ss_sum"],
    s=5,
    alpha=0.5,
    label="Prompt: matched",
    color="C0",
)
ax.set_xlabel("Date")
ax.set_ylabel("SS Objects per Visit")
ax.set_title("Solar System Objects per Visit — Prompt Processing vs DP2 DRP")
ax.set_yscale("symlog", linthresh=10)
ax.legend(markerscale=3)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True)

ax = axes[0]
ax.scatter(
    dp2_ss_nightly["date"],
    dp2_ss_nightly["n_ss_total"],
    s=15,
    alpha=0.7,
    label="DP2: predicted",
    color="C1",
)
ax.scatter(
    ss_nightly["date"],
    ss_nightly["n_ss_total"],
    s=15,
    alpha=0.7,
    label="Prompt: matched",
    color="C0",
)
ax.set_ylabel("Total SS Objects")
ax.set_title("Solar System Objects by Night — Prompt Processing vs DP2 DRP")
ax.legend()

ax = axes[1]
ax.scatter(
    dp2_ss_nightly["date"],
    dp2_ss_nightly["n_ss_median"],
    s=15,
    alpha=0.7,
    label="DP2: predicted",
    color="C1",
)
ax.scatter(
    ss_nightly["date"],
    ss_nightly["n_ss_median"],
    s=15,
    alpha=0.7,
    label="Prompt: matched",
    color="C0",
)
ax.set_ylabel("Median SS Objects / Visit")
ax.legend()

ax = axes[2]
common = ss_nightly.join(dp2_ss_nightly[["n_ss_total"]], rsuffix="_dp2")
common["recovery"] = common["n_ss_total"] / common["n_ss_total_dp2"].replace(
    0, float("nan")
)
ax.scatter(common["date"], common["recovery"], s=15, alpha=0.7, color="C2")
ax.axhline(1.0, color="gray", ls="--", alpha=0.5)
ax.set_ylabel("SS Recovery Fraction\n(Prompt matched / DP2 predicted)")
ax.set_xlabel("Night (day_obs)")

plt.tight_layout()
plt.show()